# Ophelian end-to-end demo

Same flow as `examples/demo.py`, split into cells you can run in order.

**Setup (run once):**
```bash
pip install ophelian scikit-learn
```

In [ ]:
from ophelian import Data, Deploy, Eval, Pipeline, Standalone, Train
from ophelian.envs import Auto, AutoRouterError
from ophelian.pricing import explain

## 1. Declarative pipeline

The same `Pipeline` object moves unchanged from laptop to any cloud.

In [ ]:
pipe = Pipeline(
    [
        Data(
            name="iris",
            source="synthetic://iris",
            format="synthetic",
            options={"name": "iris"},
        ),
        Train(
            name="trainer",
            framework="sklearn",
            model="sklearn.linear_model.LogisticRegression",
            data="iris",
            hyperparameters={"max_iter": 200},
        ),
        Eval(
            name="scorer",
            model="trainer",
            data="iris",
            metrics=["accuracy", "f1_macro"],
        ),
        Deploy(name="serve", model="trainer", port=8080),
    ],
    name="ophelian-demo",
)
[node.name for node in pipe.steps]

## 2. Local training run

`Standalone(local=True)` runs every step in-process — no Docker, no cloud creds.

In [ ]:
result = pipe.run(env=Standalone(local=True))
result.succeeded

## 3. Inspect step results

In [ ]:
for step in result.steps:
    print(f"{step.name:<8} {step.status}")

In [ ]:
result.step("scorer").metrics

In [ ]:
result.step("serve").info

## 4. Cost-route across AWS / GCP / Azure

`Auto(...)` picks the cheapest GPU across clouds. `dry_run=True` skips the
actual launch; `require_credentials=False` lets the router work without
any cloud SDK configured.

In [ ]:
router = Auto(
    cheapest_gpu="A100",
    regions=["us-east-1", "us-central1", "eastus"],
    dry_run=True,
    require_credentials=False,
)
decision = router.decision
print(f"{decision.quote.provider}/{decision.quote.region} "
      f"{decision.quote.instance} @ {decision.quote.hourly_usd} USD/h")
print(f"considered={len(decision.considered)}")
print(f"data_quality={decision.data_quality}")

In [ ]:
print(explain(decision))

## 5. Strict mode: `require_live=[...]`

Raises `AutoRouterError` if any required provider's price didn't come from
a live API call. Forces honest cost decisions in production code.

In [ ]:
try:
    Auto(
        cheapest_gpu="A100",
        regions=["us-east-1", "us-central1", "eastus"],
        dry_run=True,
        require_credentials=False,
        require_live=["aws", "gcp", "azure"],
        allow_live=False,
    )
except AutoRouterError as exc:
    print(f"AutoRouterError (as expected):\n  {exc}")

## Next

Swap `Standalone()` for `AWS(...)`, `GCP(...)`, `Azure(...)`, or `Auto(...)` —
the same `Pipeline` object runs on any of them.